In [1]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler

import xmlrpc.client
import datetime
import pickle
import numpy as np
import pandas as pd

In [2]:
class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("localhost", 8008), 
                            requestHandler=RequestHandler)

server.register_introspection_functions()

In [3]:
# Тест
def ping():
    return True
server.register_function(ping, 'ping')

# Время сервера
def now():
    return datetime.datetime.now()
server.register_function(now, 'now')

# Отображение строкового вида, типа и значений
def show_type(arg):
    return (str(arg), str(type(arg)), arg)
server.register_function(show_type, 'type')

# Сумма
def test_sum(a, b):
    return a + b
server.register_function(test_sum, 'sum')

# Степень
def test_pow(a, b):
    return a**b
server.register_function(test_pow, 'pow')


<function __main__.test_pow(a, b)>

In [4]:
# Пункт 4: Проверка нахождения клиента в черном списке по ФИО и дате рождения
def black_list_check(surname, name, patr, birth_date):
    try:
        frame = pd.read_csv('bad_boys2.csv', header=0, sep=',', encoding='utf-8')
        exist = any((frame['Surname'] == surname) & (frame['Name'] == name) & (frame['Patronym'] == patr) & (frame['Birth'] == birth_date) )
        if exist:
            return f"{surname} {name} {patr} {birth_date} : bad_boy"
        else:
            return f"{surname} {name} {patr} {birth_date}: good_boy"
    except FileNotFoundError:
        return "Error: 'bad_boys2.csv' not found."
    except Exception as e:
        return f"An error occurred: {e}"

server.register_function(black_list_check, 'black_list_check')

<function __main__.black_list_check(surname, name, patr, birth_date)>

In [5]:
# Пункт 5: Инверсия цвета 
def send_back_inversion(bin_data):
    
    img_arr = pickle.loads(bin_data.data)
    
    if img_arr.ndim == 3:  # Цветное изображение
        height, width, channels = img_arr.shape
        for i in range(height):
            for j in range(width):
                for k in range(channels):
                    img_arr[i, j, k] = 255 - img_arr[i, j, k]
    elif img_arr.ndim == 2:  # Монохромное изображение
        height, width = img_arr.shape
        for i in range(height):
            for j in range(width):
                img_arr[i, j] = 255 - img_arr[i, j]
    
    pimg = pickle.dumps(img_arr)        
    return xmlrpc.client.Binary(pimg)
server.register_function(send_back_inversion, 'color_inversion')

<function __main__.send_back_inversion(bin_data)>

In [6]:
# Пункт 6: Бинаризация изображения по порогу
def binarize_image(bin_data, threshold):
    
    img_arr = pickle.loads(bin_data.data)
    
    if img_arr.ndim == 3:  # Цветное
        height, width, channels = img_arr.shape
        for i in range(height):
            for j in range(width):
                # Преобразуем в оттенки серого для определения яркости
                pixel_value = (int(img_arr[i, j, 0]) + int(img_arr[i, j, 1]) + int(img_arr[i, j, 2])) / 3
                new_val = 255 if pixel_value > threshold else 0
                img_arr[i, j, 0] = new_val
                img_arr[i, j, 1] = new_val
                img_arr[i, j, 2] = new_val
    elif img_arr.ndim == 2:  # Монохромное
        height, width = img_arr.shape
        for i in range(height):
            for j in range(width):
                pixel_value = img_arr[i, j]
                img_arr[i, j] = 255 if pixel_value > threshold else 0
                
    pimg = pickle.dumps(img_arr)
    return xmlrpc.client.Binary(pimg)

server.register_function(binarize_image, 'binarize_image')

<function __main__.binarize_image(bin_data, threshold)>

In [ ]:
# Лаб2 пункт 2. Бинаризует изображение и возвращает процент пикселей выше порога.
def cloud_percent(bin_data, threshold):
    
    img_arr = pickle.loads(bin_data.data)
    if img_arr.ndim == 3:                
        height, width, channels = img_arr.shape
        # создаём пустой массив для яркостей
        gray = np.zeros((height, width), dtype=np.float64)
        for i in range(height):
            for j in range(width):
                gray[i, j] = (int(img_arr[i, j, 0]) +
                              int(img_arr[i, j, 1]) +
                              int(img_arr[i, j, 2])) / 3
    else:                               
        gray = img_arr

    # сколько пикселей выше порога
    total = gray.size                    # всего пикселей
    above = 0                            # счётчик белых
    for i in range(gray.shape[0]):
        for j in range(gray.shape[1]):
            if gray[i, j] > threshold:
                above += 1

    # процент
    percent = 100.0 * above / total
    return float(percent)


server.register_function(cloud_percent, 'cloud_percent')

In [ ]:
# Пункт 7: Разворот изображения
def flip_image_vertical(bin_data):
 
    img_arr = pickle.loads(bin_data.data)
    flipped_arr = np.zeros_like(img_arr)
    
    if img_arr.ndim == 3:
        height, width, channels = img_arr.shape
        for i in range(height):
            for j in range(width):
                flipped_arr[i, j, :] = img_arr[i, width - 1 - j, :]
    elif img_arr.ndim == 2:
        height, width = img_arr.shape
        for i in range(height):
            for j in range(width):
                flipped_arr[i, j] = img_arr[i, width - 1 - j]

    pimg = pickle.dumps(flipped_arr)
    return xmlrpc.client.Binary(pimg)

server.register_function(flip_image_vertical, 'flip_vertical')


print("Listening on port 8008...")
server.serve_forever()

Listening on port 8008...


127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:16:36] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/202